[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

# Parametric DFA: the *k*-th last input bit is 1

This notebook builds and simulates the language

$$L_k = \{w\in\{0,1\}^* : |w|\ge k \text{ and the }k\text{-th bit from the right of }w\text{ is 1}\}. $$

For example, when `k = 2`, the word `1011` is accepted because its second-last bit is `1`; `1000` is rejected. The DFA retains the most recent `k` bits. For a word shorter than `k`, its unknown earlier positions are represented by leading zeros. Therefore the DFA has exactly $2^k$ states.

The notebook uses an RTL-like split: a state register plus a pure combinational `next_state()` function. The interactive panel below is a clocked simulation with LEDs for the stored bits and for the accepting output.

In [ ]:
# Colab setup. Safe to run in ordinary Jupyter too.
%pip -q install ipywidgets networkx

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import math
import random
import ipywidgets as widgets
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, clear_output


In [ ]:
class KthLastIsOneDFA:
    """A 2^k-state DFA for: the k-th last bit read is 1."""

    def __init__(self, k):
        if not isinstance(k, int) or k < 1:
            raise ValueError('k must be a positive integer')
        self.k = k
        self.mask = (1 << k) - 1
        self.reset()

    # Combinational logic: shift in x and discard the bit older than k.
    def next_state(self, state, x):
        if x not in (0, 1):
            raise ValueError('input x must be 0 or 1')
        return ((state << 1) | x) & self.mask

    # Combinational output: MSB is the k-th last bit.
    def accepting(self, state=None):
        state = self.state if state is None else state
        return bool((state >> (self.k - 1)) & 1)

    # Sequential logic: one rising clock edge.
    def tick(self, x):
        self.state = self.next_state(self.state, x)
        self.word += str(x)
        self.cycles += 1
        return self.accepting()

    def reset(self):
        self.state = 0
        self.word = ''
        self.cycles = 0

    def bits(self, state=None):
        state = self.state if state is None else state
        return format(state, f'0{self.k}b')

    def transition_table(self):
        return [(s, self.next_state(s, 0), self.next_state(s, 1))
                for s in range(1 << self.k)]


In [ ]:
def draw_machine(dfa, target):
    """Draw the whole DFA. k <= 4 is easiest to read."""
    with target:
        clear_output(wait=True)
        n = 1 << dfa.k
        graph = nx.MultiDiGraph()
        graph.add_nodes_from(range(n))
        for state, to0, to1 in dfa.transition_table():
            graph.add_edge(state, to0, label='0')
            graph.add_edge(state, to1, label='1')

        fig_size = max(7, min(16, 2 + math.sqrt(n) * 2.2))
        fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.72))
        pos = nx.circular_layout(graph)
        colors = ['#91e69b' if dfa.accepting(s) else '#d8e0ea' for s in graph.nodes]
        borders = ['#d62728' if s == dfa.state else '#3e4a59' for s in graph.nodes]
        sizes = 1100 if n <= 16 else 700 if n <= 32 else 360
        nx.draw_networkx_nodes(graph, pos, node_color=colors, edgecolors=borders,
                               linewidths=3 if n <= 32 else 1.5, node_size=sizes, ax=ax)
        nx.draw_networkx_labels(graph, pos, labels={s: dfa.bits(s) for s in graph.nodes},
                                font_size=10 if n <= 16 else 7, ax=ax)
        nx.draw_networkx_edges(graph, pos, connectionstyle='arc3,rad=0.12',
                               arrows=True, arrowsize=15 if n <= 32 else 9, alpha=0.55, ax=ax)
        edge_labels = {(u, v, key): data['label']
                       for u, v, key, data in graph.edges(keys=True, data=True)}
        nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels,
                                     font_size=8 if n <= 16 else 6, rotate=False, ax=ax)
        ax.set_title(f'Complete DFA for k={dfa.k}: {n} states   (green = accepting; red border = current)')
        ax.axis('off')
        plt.show()

def draw_leds(dfa, target):
    with target:
        clear_output(wait=True)
        bits = dfa.bits()
        fig, ax = plt.subplots(figsize=(max(7, dfa.k * 1.05), 2.8))
        ax.set_xlim(-0.8, dfa.k + 1.7)
        ax.set_ylim(-0.4, 1.45)
        ax.axis('off')
        for i, bit in enumerate(bits):
            on = bit == '1'
            ax.add_patch(plt.Circle((i, 0.65), 0.25, color='#ffcc00' if on else '#3c4043', ec='black'))
            age = 'k-th last' if i == 0 else f'{dfa.k-i}-th last'
            ax.text(i, 1.08, age, ha='center', fontsize=9)
            ax.text(i, 0.65, bit, ha='center', va='center', fontsize=12, weight='bold')
        accepted = dfa.accepting()
        ax.add_patch(plt.Circle((dfa.k + 0.85, 0.65), 0.31,
                                color='#35c759' if accepted else '#b0b0b0', ec='black', lw=1.5))
        ax.text(dfa.k + 0.85, 1.12, 'ACCEPT', ha='center', weight='bold')
        ax.text(dfa.k + 0.85, 0.65, '1' if accepted else '0', ha='center', va='center', fontsize=13, weight='bold')
        ax.set_title(f'cycle {dfa.cycles}   |   register state = {bits}   |   input word = {dfa.word or "ε"}')
        plt.show()


In [ ]:
# Interactive Colab control panel
k_picker = widgets.IntSlider(value=3, min=1, max=6, step=1, description='k:', continuous_update=False)
zero = widgets.Button(description='Clock in 0')
one = widgets.Button(description='Clock in 1', button_style='primary')
random_word = widgets.Button(description='Random 12-bit word')
reset = widgets.Button(description='Reset')
status = widgets.HTML()
led_out = widgets.Output()
graph_out = widgets.Output()

dfa = KthLastIsOneDFA(k_picker.value)

def refresh(redraw_graph=False):
    verdict = '<b style="color:green">ACCEPT</b>' if dfa.accepting() else '<b style="color:crimson">REJECT</b>'
    status.value = (f'<b>{1 << dfa.k} states</b> &nbsp; | &nbsp; current state: <code>{dfa.bits()}</code>'
                    f' &nbsp; | &nbsp; result: {verdict}')
    draw_leds(dfa, led_out)
    if redraw_graph:
        draw_machine(dfa, graph_out)

def clock(x):
    dfa.tick(x)
    refresh()

zero.on_click(lambda _: clock(0))
one.on_click(lambda _: clock(1))

def do_reset(_):
    dfa.reset()
    refresh()

def do_random(_):
    dfa.reset()
    for _ in range(12):
        dfa.tick(random.randint(0, 1))
    refresh()

def set_k(change):
    global dfa
    dfa = KthLastIsOneDFA(change['new'])
    refresh(redraw_graph=True)

reset.on_click(do_reset)
random_word.on_click(do_random)
k_picker.observe(set_k, names='value')

display(widgets.VBox([
    widgets.HTML('<h3>Clocked DFA simulator</h3>'),
    k_picker,
    widgets.HBox([zero, one, random_word, reset]),
    status, led_out, graph_out
]))
refresh(redraw_graph=True)


## Why the state count is exponential

At the end of the word, the machine must determine the bit that arrived exactly `k` clock cycles ago. Thus it must preserve all `k` recent bits. There are $2^k$ possible bit patterns, so the construction has $2^k$ states. The accepting states are precisely those whose leftmost displayed state bit is `1`; exactly half of the states accept.

Try `k=1`, `k=2`, `k=3`, then `k=5` or `k=6`. The graph becomes visually dense quite quickly, while the LED panel remains useful.